# GenieX NPU Inference Examples

This notebook demonstrates how to use the GenieX for various AI inference tasks on NPU devices, including:

- **LLM (Large Language Model)**: Text generation and conversation
- **VLM (Vision Language Model)**: Multimodal understanding and generation
## Prerequisites

### 1. Install the correct Python version

If you prefer, we also offer a video tutorial for the installation. Check it out [here](https://www.youtube.com/watch?v=ziXKPRX0Ufo).

GenieX requires **Python 3.11 – 3.13 (ARM64 build)** on Windows ARM.
Please **download and install the official ARM64 Python** from the [python-3.11.1-arm64.exe](https://www.python.org/ftp/python/3.11.1/python-3.11.1-arm64.exe). Make sure you read the instructions below carefully before proceeding.

> ❗ **IMPORTANT**: Make sure you select "Add python.exe to PATH" on the first screen of the installation wizard.

> 🛑 Make sure you restart the terminal or your IDE after installation.

> ⚠️ Do **not** use Conda or x86 builds — they are incompatible with native ARM64 binaries. If you are in a conda environment, run `conda deactivate` first.

Verify the installation:

In case your environment path gets overriden by some environment manager, we recommend you to run the following commands to restore PATH variable from system settings.
```powershell
$systemPath = [Environment]::GetEnvironmentVariable('Path', 'Machine')
$userPath   = [Environment]::GetEnvironmentVariable('Path', 'User')
$env:Path   = "$userPath;$systemPath"
```

Then verify your python executable has the correct architecture and version (3.11 - 3.13)
```sh
python -c "import sys, platform; print(f'Python version: {sys.version}')"
```

Your output should look like:

> Python version: 3.11.0 (main, Oct 24 2022, 18:15:22) [MSC v.1933 64 bit (ARM64)]

Expected output must contain version `3.11.0` and architecture `ARM64`.

If it does show `AMD64` or incorrect version, try the following:

- (If you have conda installed) Run `conda deactivate` to deactivate the current conda environment.
- (If your `python` executable points to the x86 version) You may need to make the ARM64 Python come before the x86 Python in your PATH.
  - Hit the `Win` key, and type `env`, and hit Enter to select `Edit the system environment variables` setting.
  - Click on `Environment Variables...` button.
  - Select `Path` and click `Edit...`.
  - Find your ARM64 Python installation path, and move it to the top of the list.
  - Hit `OK` for several times to close all the dialogs and save the changes.
- (If you forgot to select "Add python.exe to PATH" on the first screen of the installation wizard)
  - Run the installation wizard again, follow the instructions to remove the current installation, and then reinstall from the Wizard. Make sure to select "Add python.exe to PATH" this time.

---

### 2. Create and activate a virtual environment

`cd` to the current project root directory `cd path/to/geniex`.

```sh
python -m venv geniex-env
geniex-env\Scripts\activate
```

---

### 3. Install the Geniex

```bash
pip install geniex -v
```

---

### 4. Select the venv as your Jupyter Notebook kernel

- Depending the editor you are using, the way to change kernel might be different. For Cursor / VS Code, they are located at the top right corner of you code window.
- Look for and select the `geniex-env`, or the custom virtual environment you have created. The kernel should automatically reload in most IDEs.


### Verification of Kernel

Run the following code to ensure you have the right kernel running.


In [ ]:
import sys
import platform

# ANSI color codes
RED = "\033[91m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
BOLD = "\033[1m"
RESET = "\033[0m"

min_ver = (3, 11)
max_ver = (3, 13)
current_ver = sys.version_info
arch = platform.machine()

if not (min_ver <= (current_ver.major, current_ver.minor) < max_ver) or arch.lower() != "arm64":
    print("\n" + "=" * 80)
    print(f"{BOLD}{RED}WARNING: Your Python version or architecture is not compatible.{RESET}")
    print(f"Detected version: {current_ver.major}.{current_ver.minor}, architecture: {arch}")
    print(f"{YELLOW}Required: Python 3.11 - 3.13 & architecture 'arm64'.{RESET}")
    print("=" * 80)
    print(f"{RED}DO NOT continue to the following code!{RESET}\n")
    print("To install arm64 Python:")
    print("  - Download Python 3.11-3.13 for arm64 from https://www.python.org/downloads/")
    print("  - Install and verify by running: python3 --version and python3 -c 'import platform; print(platform.machine())'")
    print("  - Launch Jupyter and make sure to select the arm64 Python kernel in 'Kernel > Change kernel'.")
    sys.exit(1)
else:
    print(f"{GREEN}[VERIFICATION PASSED] Python version and architecture are correct. You may continue to the following sections.{RESET}")


## Authentication Setup

Before running any examples, you need to set up your NexaAI authentication token.

### Set Token in Code

Replace `"YOUR_NEXA_TOKEN_HERE"` with your actual NexaAI token from [https://sdk.nexa.ai/](https://sdk.nexa.ai/):


In [ ]:
import os

# Replace "YOUR_NEXA_TOKEN_HERE" with your actual token from https://sdk.nexa.ai/
os.environ["NEXA_TOKEN"] = "YOUR_NEXA_TOKEN_HERE"
# Suppress HF warnings
os.environ["HF_HUB_VERBOSITY"] = "error"

assert os.environ.get("NEXA_TOKEN", "").startswith(
    "key/"), "ERROR: NEXA_TOKEN must start with 'key/'. Please check your token."


## 1. LLM (Large Language Model) NPU Inference

Using NPU-accelerated large language models for text generation and conversation. Llama3.2-3B-NPU-Turbo is specifically optimized for NPU.


In [ ]:
import io
import os

from nexaai import LLM, GenerationConfig, ModelConfig, LlmChatMessage


def llm_npu_example():
    """LLM NPU inference example"""
    print("=== LLM NPU Inference Example ===")

    # Model configuration

    # Use huggingface Repo ID
    model_name = "NexaAI/Llama3.2-3B-NPU-Turbo"
    # Alternatively, use local path
    # model_name = os.path.expanduser(r"~\.cache\nexa.ai\nexa_sdk\models\NexaAI\Llama3.2-3B-NPU-Turbo\weights-1-3.nexa")

    plugin_id = "npu"
    max_tokens = 100
    system_message = "You are a helpful assistant."

    print(f"Loading model: {model_name}")
    print(f"Using plugin: {plugin_id}")

    # Create model instance
    config = ModelConfig()
    llm = LLM.from_(model=model_name, plugin_id=plugin_id, config=config)

    # Create conversation history
    conversation = [LlmChatMessage(role="system", content=system_message)]

    # Example conversations
    test_prompts = [
        "What is artificial intelligence?",
        "Explain the benefits of on-device AI processing.",
        "How does NPU acceleration work?"
    ]

    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n--- Conversation {i} ---")
        print(f"User: {prompt}")

        # Add user message
        conversation.append(LlmChatMessage(role="user", content=prompt))

        # Apply chat template
        formatted_prompt = llm.apply_chat_template(conversation)

        # Generate response
        print("Assistant: ", end="", flush=True)
        response_buffer = io.StringIO()

        gen = llm.generate_stream(formatted_prompt, GenerationConfig(max_tokens=max_tokens))
        result = None
        try:
            while True:
                token = next(gen)
                print(token, end="", flush=True)
                response_buffer.write(token)
        except StopIteration as e:
            result = e.value

        # Get profiling data
        if result and hasattr(result, "profile_data") and result.profile_data:
            print(f"\n{result.profile_data}")

        # Add assistant response to conversation history
        conversation.append(LlmChatMessage(role="assistant", content=response_buffer.getvalue()))
        print("\n" + "=" * 50)


llm_npu_example()


## 2. VLM (Vision Language Model) NPU Inference

Using NPU-accelerated vision language models for multimodal understanding and generation. OmniNeural-4B supports joint processing of images and text.


In [ ]:
import os
import io

from nexaai import (
    GenerationConfig,
    ModelConfig,
    VlmChatMessage,
    VlmContent,
)
from nexaai.vlm import VLM


def vlm_npu_example():
    """VLM NPU inference example"""
    print("=== VLM NPU Inference Example ===")

    # Model configuration

    # Use huggingface repo ID
    model_name = "NexaAI/OmniNeural-4B"
    # Alternatively, use local path
    # model_name = os.path.expanduser(r"~\.cache\nexa.ai\nexa_sdk\models\NexaAI\OmniNeural-4B\weights-1-8.nexa")

    plugin_id = "npu"
    max_tokens = 100
    system_message = "You are a helpful assistant that can understand images and text."
    image_path = '/your/image/path'  # Replace with actual image path if available

    print(f"Loading model: {model_name}")
    print(f"Using plugin: {plugin_id}")

    # Check for image existence
    if not (image_path and os.path.exists(image_path)):
        print(
            f"\033[93mWARNING: The specified image_path ('{image_path}') does not exist or was not provided. Multimodal prompts will not include image input.\033[0m")

    # Create model instance
    config = ModelConfig()
    vlm = VLM.from_(model=model_name, config=config, plugin_id=plugin_id)

    # Create conversation history
    conversation = [
        VlmChatMessage(
            role="system",
            contents=[VlmContent(type="text", text=system_message)]
        )
    ]

    # Example multimodal conversations
    test_cases = [
        {
            "text": "What do you see in this image?",
            "image_path": image_path
        }
    ]

    for i, case in enumerate(test_cases, 1):
        print(f"\n--- Multimodal Conversation {i} ---")
        print(f"User: {case['text']}")

        # Build message content
        contents = []
        if case['text']:
            contents.append(VlmContent(type="text", text=case['text']))

        # Add image content if available
        if case['image_path'] and os.path.exists(case['image_path']):
            contents.append(VlmContent(type="image", text=case['image_path']))
            print(f"Including image: {case['image_path']}")

        # Add user message
        conversation.append(VlmChatMessage(role="user", contents=contents))

        # Apply chat template
        formatted_prompt = vlm.apply_chat_template(conversation)

        # Generate response
        print("Assistant: ", end="", flush=True)
        response_buffer = io.StringIO()

        # Prepare image and audio paths
        image_paths = [case['image_path']] if case['image_path'] and os.path.exists(case['image_path']) else None
        audio_paths = None

        gen = vlm.generate_stream(
            formatted_prompt,
            config=GenerationConfig(
                max_tokens=max_tokens,
                image_paths=image_paths,
                audio_paths=audio_paths
            )
        )
        result = None
        try:
            while True:
                token = next(gen)
                print(token, end="", flush=True)
                response_buffer.write(token)
        except StopIteration as e:
            result = e.value

        # Get profiling data
        if result and hasattr(result, "profile_data") and result.profile_data:
            print(f"\n{result.profile_data}")

        # Add assistant response to conversation history
        conversation.append(
            VlmChatMessage(
                role="assistant",
                contents=[
                    VlmContent(type="text", text=response_buffer.getvalue())
                ]
            )
        )
        print("\n" + "=" * 50)


vlm_npu_example()
